In [2]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
from pathlib import Path
import librosa
from pyrubberband import timemap_stretch
from fastdtw import fastdtw # type: ignore

In [3]:
from perturbations.data.audio_io import load_performances, load_audio
from perturbations.consts import SAMPLE_RATE
from perturbations.feature_extract import load_performances_features, calculate_rhythm_features

/home/toni/projects/perturbations/audio


[   INFO   ] MusicExtractorSVM: no classifier models were configured by default


In [7]:
studio_path = Path("../audio/studio/ween/ween-rosesarefree_studio.flac")
studio_audio = load_audio(studio_path)

studio_verse1_beginning = int(4*SAMPLE_RATE)
studio_verse1_end = int(46*SAMPLE_RATE) 

studio_audio_trimmed = studio_audio[ 
  studio_verse1_beginning : 
  studio_verse1_end 
]

Audio(data=studio_audio_trimmed, rate=SAMPLE_RATE)

In [12]:
studio_rhythm = calculate_rhythm_features(studio_audio_trimmed)

print(studio_rhythm.beats)
print(studio_rhythm.bpm)

[ 0.6849887  1.3931973  2.113016   2.8328345  3.4829931  4.133152
  4.7833104  5.4334693  6.083628   6.710567   7.3607254  8.010884
  8.649433   9.287982   9.938141  10.588299  11.238458  11.8653965
 12.515555  13.165714  13.804263  14.442812  15.09297   15.743129
 16.370068  17.020226  17.670385  18.308933  18.947483  19.597641
 20.2478    20.897959 ]
93.01091003417969


In [10]:
perf_path = Path("../audio/ween_rosesarefree/raw/ween2003-10-03d1t17.flac")
perf_audio = load_audio(perf_path)

perf_verse1_beginning = int(26*SAMPLE_RATE)
perf_verse1_end = int(68*SAMPLE_RATE) 

perf_audio_trimmed = perf_audio[ 
  perf_verse1_beginning : 
  perf_verse1_end 
]

Audio(data=perf_audio_trimmed, rate=SAMPLE_RATE)

In [14]:
perf_rhythm = calculate_rhythm_features(perf_audio_trimmed)

print(perf_rhythm.beats)
print(perf_rhythm.bpm)

[ 0.7198186  1.4628571  2.1362357  2.8096144  3.471383   4.133152
  4.8065305  5.491519   6.141678   6.803447   7.465215   8.138594
  8.800363   9.4621315 10.13551   10.808888  11.470657  12.132426
 12.805805  13.467573  14.129342  14.802721  15.476099  16.149479
 16.799637  17.461405  18.134785  18.796553  19.469933  20.14331
 20.805079 ]
89.97106170654297


In [21]:
distance, path = fastdtw(
    studio_rhythm.beats.reshape(-1, 1),
    perf_rhythm.beats.reshape(-1, 1)
)
print(distance)
print(path)

4.957457721233368
[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 8), (9, 9), (10, 10), (11, 11), (12, 12), (13, 13), (14, 14), (15, 15), (16, 16), (17, 17), (18, 18), (19, 19), (20, 20), (21, 20), (22, 21), (23, 22), (24, 23), (25, 24), (26, 25), (27, 26), (28, 27), (29, 28), (30, 29), (31, 30)]


In [18]:
time_map = [
    (
        int(studio_rhythm.beats[i] * SAMPLE_RATE),
        int(perf_rhythm.beats[j] * SAMPLE_RATE)
    )
    for i, j in path
]

time_map.append((len(studio_audio_trimmed), len(perf_audio_trimmed)))



In [23]:

studio_warped = timemap_stretch(studio_audio_trimmed, SAMPLE_RATE, time_map)

In [24]:
studio_warped_rhythm = calculate_rhythm_features(studio_warped)

In [ ]:
print(studio_warped_rhythm.bpm) # 91.63496398925781
print(perf_rhythm.bpm) # 89.97106170654297
print(studio_rhythm.bpm) # 93.01091003417969

91.63496398925781
89.97106170654297
93.01091003417969


In [28]:
distance_warped, path_warped = fastdtw(
    perf_rhythm.beats.reshape(-1, 1),
    studio_warped_rhythm.beats.reshape(-1, 1)
) # should be lower, but has gotten higher?

print(distance_warped)
print(path_warped)

7.929612457752228
[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 8), (9, 9), (10, 10), (11, 11), (12, 12), (13, 13), (14, 14), (15, 15), (16, 16), (17, 17), (17, 18), (18, 19), (19, 20), (20, 21), (21, 22), (22, 23), (23, 24), (24, 25), (25, 26), (26, 27), (27, 28), (28, 29), (29, 30), (30, 30)]


In [29]:
Audio(data=studio_warped, rate=SAMPLE_RATE)